# Training w/ controls!

Come up with labeling stratergy to predict controls!
We want to prioritize disease embedding learning and to seperate disease from control conditions.

In [1]:
# imports
import sys
import pandas as pd
import numpy as np
import scanpy as sc
sys.path.append("../../")
from src.utils import utils as u


adata = sc.read_h5ad("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-12-02/data.h5ad")

# get Disease Ontology graph
do_g = u.load_do_graph()

# get sanchez IC
doid_2_ic = u.get_sanchez_ic(do_g)


Number of DO leaves: 9018


In [6]:
import importlib
importlib.reload(u)

<module 'src.utils.utils' from '/aloy/home/ddalton/projects/scGPT_playground/notebooks/exp/../../src/utils/utils.py'>

In [2]:

#! IMPORTANT - FIX CONTROL CLASS
adata.obs["do_id"] = adata.obs["doid_id"].astype(str)
# get nodes
_class_nodes = u.get_lvl1_nodes(do_g)
# _class_nodes = u.get_n_lowest_ic_nodes(doid_2_ic, 50)
# benchmark class nodes - use the sames as in the single classifier benchmark
# _class_nodes = adata.obs["do_id"].unique().tolist()

# Generate multilabel vectors for class nodes
Y_multilabel, _class_nodes = u.generate_multilabel_vectors(
    adata, do_g, _class_nodes
)
print(
    f"Generated multilabel vectors for class nodes with shape {Y_multilabel.shape}"
)

# Clean multilabel vectors by removing nodes with no samples
Y_multilabel, _class_nodes = u.clean_multilabel_vectors(Y_multilabel, _class_nodes)
print(
    f"Cleaned multilabel vectors for class nodes with shape {Y_multilabel.shape}"
)

# Check the multilabel vector for class nodes
u.check_multilabel_vector(Y_multilabel, _class_nodes, do_g)

# get doids and class names
Y_multilabel_doid, Y_multilabel_name = u.get_multilabel_data(
    Y_multilabel, _class_nodes, do_g
)

# add multilabel vectors to adata
#! Note: we can add and extract arrays from the data frame - but not save them !
adata.obs = u.add_multilabel_to_adata(
    adata, Y_multilabel, Y_multilabel_doid, Y_multilabel_name
)


Root node: DOID:4 - disease
Nº Level 1 nodes: 8
Generated multilabel vectors for class nodes with shape (46136, 9)
Cleaned multilabel vectors for class nodes with shape (46136, 9)
19131 samples	Node Control
620 samples	Node DOID:0014667 - disease of metabolism
826 samples	Node DOID:0050117 - disease by infectious agent
77 samples	Node DOID:0080015 - physical disorder
11918 samples	Node DOID:14566 - disease of cellular proliferation
946 samples	Node DOID:150 - disease of mental health
498 samples	Node DOID:225 - syndrome
1650 samples	Node DOID:630 - genetic disease
22854 samples	Node DOID:7 - disease of anatomical entity
Nº of samples with only one label: 33910
Nº of samples with +1 labels: 12226
Max nº of labels per sample: 3


In [3]:
Y_multilabel

array([[1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
a = ["A", "B", "D"]


# define class nodes
nodes = sorted(a)


# check for presence of controls
if "Control" in adata.obs["do_id"].unique():
    nodes.insert(0, "Control")  # add Control as a class node



In [10]:
Y_multilabel.sum(axis=1)

array([0, 0, 0, ..., 1, 1, 1])

In [4]:
adata.obs

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,doid_disease
0,DSA00004.GSM7009973.Control,GSE224022,GSE224022,369,369,DSA00004,Retina,19402,Control,Control,Retinoblastoma,RNA-Seq,DOID:768,Control,Control
1,DSA00004.GSM7009974.Control,GSE224022,GSE224022,369,369,DSA00004,Retina,19402,Control,Control,Retinoblastoma,RNA-Seq,DOID:768,Control,Control
2,DSA00004.GSM7009976.Control,GSE224022,GSE224022,369,369,DSA00004,Retina,19402,Control,Control,Retinoblastoma,RNA-Seq,DOID:768,Control,Control
3,DSA00004.GSM7009977.Control,GSE224022,GSE224022,369,369,DSA00004,Retina,19402,Control,Control,Retinoblastoma,RNA-Seq,DOID:768,Control,Control
4,DSA00004.GSM7009978.Case,GSE224022,GSE224022,369,369,DSA00004,Retina,19402,Retinoblastoma,Retinoblastoma,Retinoblastoma,RNA-Seq,DOID:768,DOID:768,retinoblastoma
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46131,DSA10149.GSM5008706.Case,GSE164376,GSE164376,579,579,DSA10149,nan,19402,Schizophrenia,Schizophrenia,Schizophrenia,RNA-Seq,DOID:5419,DOID:5419,schizophrenia
46132,DSA10149.GSM5008707.Case,GSE164376,GSE164376,579,579,DSA10149,nan,19402,Schizophrenia,Schizophrenia,Schizophrenia,RNA-Seq,DOID:5419,DOID:5419,schizophrenia
46133,DSA10149.GSM5008708.Case,GSE164376,GSE164376,579,579,DSA10149,nan,19402,Schizophrenia,Schizophrenia,Schizophrenia,RNA-Seq,DOID:5419,DOID:5419,schizophrenia
46134,DSA10149.GSM5008709.Case,GSE164376,GSE164376,579,579,DSA10149,nan,19402,Schizophrenia,Schizophrenia,Schizophrenia,RNA-Seq,DOID:5419,DOID:5419,schizophrenia
